In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import torch.utils.model_zoo as model_zoo
from dataset import CityscapesDataset, SimpleTransform
from model import CombinedLoss, CBAM

# ==================== 1. ResNet & DPAI Encoder Components ====================

class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return self.relu(out)

class AdaptiveBidirectionalInteraction(nn.Module):
    def __init__(self, s_channels, d_channels, reduction=16):
        super(AdaptiveBidirectionalInteraction, self).__init__()
        self.s_conv7x7 = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.s_to_d_mlp = nn.Sequential(
            nn.Linear(1, d_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(d_channels // reduction, d_channels)
        )
        self.d_shared_mlp = nn.Sequential(
            nn.Linear(d_channels, d_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(d_channels // reduction, d_channels, bias=False)
        )
        mid_channels = 256
        self.d_to_s_conv = nn.Sequential(
            nn.Conv2d(d_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, 1, kernel_size=1, bias=False),
            nn.Sigmoid()
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, Fs, Fd):
        avg_p = F.adaptive_avg_pool2d(Fd, 1).view(Fd.size(0), -1)
        max_p = F.adaptive_max_pool2d(Fd, 1).view(Fd.size(0), -1)
        d_stats = self.d_shared_mlp(avg_p) + self.d_shared_mlp(max_p)
        Ac = self.sigmoid(d_stats)

        s_avg = torch.mean(Fs, dim=1, keepdim=True)
        s_max, _ = torch.max(Fs, dim=1, keepdim=True)
        s_desc = torch.cat([s_avg, s_max], dim=1)

        Gds = self.d_to_s_conv(Fd * Ac.view(Fd.size(0), Fd.size(1), 1, 1))
        Gds_up = F.interpolate(Gds, size=(Fs.size(2), Fs.size(3)), mode='bilinear', align_corners=False)

        s_desc_fused = s_desc * Gds_up
        As = self.sigmoid(self.s_conv7x7(s_desc_fused))
        Fs_out = Fs * As

        s_guide_vec = F.adaptive_avg_pool2d(As, 1).view(As.size(0), -1)
        Gsd = self.s_to_d_mlp(s_guide_vec)

        final_Ac = self.sigmoid(d_stats + Gsd).view(Fd.size(0), Fd.size(1), 1, 1)
        Fd_out = Fd * final_Ac

        return Fs_out, Fd_out

# ==================== 2. UNet Decoder Components ====================

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)

class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNetDecoder(nn.Module):
    def __init__(self, encoder_channels=[64, 256, 512, 1024, 2048],
                 decoder_channels=[256, 128, 64, 32], n_classes=19, bilinear=True):
        super(UNetDecoder, self).__init__()
        enc_chs = encoder_channels[::-1]
        self.up1 = UpBlock(enc_chs[0] + enc_chs[1], decoder_channels[0], bilinear)
        self.up2 = UpBlock(decoder_channels[0] + enc_chs[2], decoder_channels[1], bilinear)
        self.up3 = UpBlock(decoder_channels[1] + enc_chs[3], decoder_channels[2], bilinear)
        self.up4 = UpBlock(decoder_channels[2] + enc_chs[4], decoder_channels[3], bilinear)
        self.outc = nn.Conv2d(decoder_channels[3], n_classes, kernel_size=1)

    def forward(self, features):
        x0, x1, x2, x3, x4 = features
        d4 = self.up1(x4, x3)
        d3 = self.up2(d4, x2)
        d2 = self.up3(d3, x1)
        d1 = self.up4(d2, x0)
        return self.outc(d1)

# ==================== 3. Complete DPAI-UNet Model ====================

class DPAI_UNet_Segmentation(nn.Module):
    def __init__(self, block=Bottleneck, layers=[3, 4, 6, 3], num_classes=19, pretrained=False):
        super(DPAI_UNet_Segmentation, self).__init__()
        self.inplanes = 64

        # Encoder 前缀
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet 层级
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.cbam1 = CBAM(256)
        self.cbam2 = CBAM(512)
        self.cbam3 = CBAM(1024)
        self.cbam4 = CBAM(2048)

        # DPAI 双向交互模块 (Heavy / Light)
        self.interaction_heavy = AdaptiveBidirectionalInteraction(512, 2048)
        self.interaction_light = AdaptiveBidirectionalInteraction(256, 1024)
        self.x0_enhance = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64)
        )
        # ==============================================================================

        # U-Net Decoder (融合深浅层特征)
        self.unet_decoder = UNetDecoder(
            encoder_channels=[64, 256, 512, 1024, 2048],
            decoder_channels=[256, 128, 64, 32],
            n_classes=num_classes
        )

        self._initialize_weights()
        if pretrained:
            self._load_pretrained_weights()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1); m.bias.data.zero_()

    def _load_pretrained_weights(self):
        try:
            pre_dict = model_zoo.load_url('https://download.pytorch.org/models/resnet50-19c8e357.pth')
            model_dict = self.state_dict()
            pre_dict = {k: v for k, v in pre_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
            model_dict.update(pre_dict)
            self.load_state_dict(model_dict)
            print("Successfully loaded pre-trained ResNet-50 weights in Encoder!")
        except Exception as e:
            print("Failed to load pre-trained weights:", e)

    def forward(self, x, size=None):
        # ------ 1. Encoder (特征提取) ------
        x0 = self.relu(self.bn1(self.conv1(x)))
        x_low = self.maxpool(x0)

        x1 = self.layer1(x_low)
        x1 = self.cbam1(x1)
        x2 = self.layer2(x1)  # 浅层 Fs
        x2 = self.cbam2(x2)
        x3 = self.layer3(x2)
        x3 = self.cbam3(x3)
        x4 = self.layer4(x3)  # 深层 Fd
        x4 = self.cbam4(x4)

        # ------ 2. DPAI 模块 (适应性交互) ------
        x0_processed = self.x0_enhance(x0)
        x2_enhanced, x4_enhanced = self.interaction_heavy(x2, x4)
        x1_enhanced, x3_enhanced = self.interaction_light(x1, x3)

        # 残差相加，得到融合并增强后的特征
        x0_fused = self.relu(x0 + x0_processed)  # 经典的 ResNet 残差处理！
        x1_fused = self.relu(x1 + x1_enhanced)
        x2_fused = self.relu(x2 + x2_enhanced)
        x3_fused = self.relu(x3 + x3_enhanced)
        x4_fused = self.relu(x4 + x4_enhanced)

        # ------ 3. UNet Decoder (特征解码) ------
        # 使用 UNet Decoder 进行逐级上采样并拼接
        features = [x0_fused, x1_fused, x2_fused, x3_fused, x4_fused]
        decoder_out = self.unet_decoder(features)

        # ------ 4. Final Upsampling ------
        # 由于 Decoder 出来的尺寸是原图的 1/2，需要最后插值回原图大小
        target_size = size if size is not None else (x.size(2), x.size(3))
        return F.interpolate(decoder_out, size=target_size, mode='bilinear', align_corners=False)


# ==================== 4. 数据处理与损失函数 (从 DPAI_FPN.ipynb 拷贝) ====================
from torch.utils.data import Dataset, DataLoader

class CityscapesDataset(Dataset):
    """Cityscapes数据集加载器"""
    CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic light',
               'traffic sign', 'vegetation', 'terrain', 'sky', 'person', 'rider', 'car',
               'truck', 'bus', 'train', 'motorcycle', 'bicycle']

    LABEL_ID_MAPPING = {7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
                        23: 10, 24: 11, 25: 12, 26: 13, 27: 14, 28: 15, 31: 16, 32: 17, 33: 18}

    def __init__(self, root, split='train', transform=None, target_size=(1024, 512)):
        self.root, self.split, self.transform, self.target_size = root, split, transform, target_size
        self.images = self._get_image_paths()
        self.targets = self._get_target_paths()

    def _get_image_paths(self):
        image_dir = os.path.join(self.root, 'leftImg8bit', self.split)
        images = []
        for city in os.listdir(image_dir):
            city_dir = os.path.join(image_dir, city)
            for img_name in os.listdir(city_dir):
                if img_name.endswith('_leftImg8bit.png'): images.append(os.path.join(city_dir, img_name))
        return sorted(images)

    def _get_target_paths(self):
        target_dir = os.path.join(self.root, 'gtFine', self.split)
        targets = []
        for img_path in self.images:
            img_name = os.path.basename(img_path)
            base_name = img_name.replace('_leftImg8bit.png', '')
            city = os.path.basename(os.path.dirname(img_path))
            target_name = f'{base_name}_gtFine_labelIds.png'
            targets.append(os.path.join(target_dir, city, target_name))
        return sorted(targets)

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert('RGB')
        target = Image.open(self.targets[idx])
        orig_size = image.size[::-1]
        image = image.resize(self.target_size, Image.BILINEAR)
        target = target.resize(self.target_size, Image.NEAREST)
        if self.transform: image, target = self.transform(image, target)
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
        target = self._remap_labels(np.array(target))
        return image, target, orig_size, self.images[idx]

    def _remap_labels(self, target):
        remapped = np.full_like(target, 255)
        for old, new in self.LABEL_ID_MAPPING.items(): remapped[target == old] = new
        return torch.from_numpy(remapped).long()

class CombinedLoss(nn.Module):
    """组合损失"""
    def __init__(self, num_classes=19, ignore_index=255):
        super(CombinedLoss, self).__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.num_classes = num_classes

    def forward(self, pred, target):
        ce_loss = self.ce(pred, target)
        return ce_loss

# ==================== 5. 训练器与主逻辑 ====================

class SegmentationTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, scheduler, device, save_dir='checkpoints_unet'):
        self.model, self.train_loader, self.val_loader = model, train_loader, val_loader
        self.criterion, self.optimizer, self.scheduler, self.device = criterion, optimizer, scheduler, device
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.train_losses, self.val_losses, self.val_miou_history = [], [], []
        self.epoch_loss_history = []
        self.batch_loss_history = []
        self.best_miou, self.epoch = 0, 0

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        running_loss = 0.0
        log_interval = 50

        for batch_idx, (images, targets, _, _) in enumerate(self.train_loader):
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            loss = self.criterion(outputs, targets)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            loss_val = loss.item()
            total_loss += loss_val
            running_loss += loss_val

            # 每 50 个 Batch 记录一次平均 Loss
            if (batch_idx + 1) % log_interval == 0:
                avg_batch_loss = running_loss / log_interval
                self.batch_loss_history.append({
                    'epoch': self.epoch + 1,
                    'batch': batch_idx + 1,
                    'avg_loss': avg_batch_loss
                })
                print(f"  Epoch {self.epoch+1}, Batch {batch_idx+1}, Group Avg Loss: {avg_batch_loss:.4f}")
                running_loss = 0.0
        return total_loss / len(self.train_loader)

    @torch.no_grad()
    def validate(self):
        self.model.eval()
        total_loss, confusion_matrix = 0, np.zeros((19, 19))
        for images, targets, _, _ in self.val_loader:
            images, targets = images.to(self.device), targets.to(self.device)
            outputs = self.model(images)
            total_loss += self.criterion(outputs, targets).item()
            preds = outputs.argmax(dim=1).cpu().numpy()
            for t, p in zip(targets.cpu().numpy(), preds):
                mask = (t != 255)
                label = 19 * t[mask].astype('int') + p[mask]
                confusion_matrix += np.bincount(label, minlength=19**2).reshape(19, 19)
        iu = np.diag(confusion_matrix) / (confusion_matrix.sum(axis=1) + confusion_matrix.sum(axis=0) - np.diag(confusion_matrix) + 1e-10)
        return total_loss / len(self.val_loader), np.mean(iu)

    def train(self, num_epochs):
        for epoch in range(num_epochs):
            self.epoch = epoch
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            avg_epoch_loss = self.train_epoch()
            val_loss, val_miou = self.validate()

            self.epoch_loss_history.append({
                'epoch': epoch + 1,
                'train_loss': avg_epoch_loss,
                'val_loss': val_loss,
                'miou': val_miou
            })

            print(f"--- Epoch {epoch+1} Summary ---")
            print(f"Average Train Loss: {avg_epoch_loss:.6f}")
            print(f"Average Val Loss:   {val_loss:.6f}")
            print(f"mIoU:               {val_miou:.4f}")
            if val_miou > self.best_miou:
                self.best_miou = val_miou
                # =============== 【修改核心 3：智能剥离多卡前缀，保证保存干净的权重】 ===============
                model_to_save = self.model.module if hasattr(self.model, 'module') else self.model

                # 注意：保存前确保目标文件夹存在，原代码中 'dpai_unet' 目录如果不存在会报错
                os.makedirs(os.path.join(self.save_dir, 'dpai_unet'), exist_ok=True)

                torch.save(model_to_save.state_dict(), os.path.join(self.save_dir, 'dpai_unet/dpai_unet.pth'))
                # ===================================================================================
                print("保存最佳模型!")

# ==================== 6. 运行脚本 ====================

def main():
    # 路径配置
    root_dir = r"/home/ps/gzx/D2D_2.0"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 实例化改进模型 (DPAI_UNet)
    model = DPAI_UNet_Segmentation(num_classes=19, pretrained=True)
        # =============== 【修改核心 1：加入多卡封装】 ===============
    if torch.cuda.device_count() > 1:
        print(f"�� 检测到 {torch.cuda.device_count()} 张显卡，开启 nn.DataParallel 双卡加速！")
        model = nn.DataParallel(model)
    model = model.to(device)
    # ==========================================================

    train_dataset = CityscapesDataset(root_dir, 'train')
    val_dataset = CityscapesDataset(root_dir, 'val')

    print(f"训练集图片数量: {len(train_dataset)}")
    print(f"验证集图片数量: {len(val_dataset)}")

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)

    print(f"训练集每个epoch的batch数量: {len(train_loader)}")
    print(f"验证集每个epoch的batch数量: {len(val_loader)}")

    criterion = CombinedLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    trainer = SegmentationTrainer(model, train_loader, val_loader, criterion, optimizer, None, device, save_dir='checkpoints_dpai_unet')
    trainer.train(num_epochs=50)

# if __name__ == "__main__":
#     main()


In [ ]:
import matplotlib
# 设置中文字体支持
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

# --- ��������������� ---
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
# -----------------------


def load_unet_model_weight(model, weight_path, device):
    """加载之前跑出来的最好的 .pth 权重文件"""
    if not os.path.exists(weight_path):
        print(f"❌ 找不到权重文件: {weight_path}")
        return model

    # 1. 挂载权重
    state_dict = torch.load(weight_path, map_location=device)

    # 2. 清理多显卡训练时可能残存的并行 'module.' 前缀 (剥离操作)
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        new_state_dict[name] = v

    model.load_state_dict(new_state_dict, strict=True)
    model.to(device)
    model.eval()  # 设置为推理模式至关重要（关闭 Dropout 和 BN 特性）
    print("✅ 预训练权重装填成功！")
    return model

def predict_and_visualize(model, image_path, device, target_size=(1024, 512), save_dir='visual_results'):
    """接受一张图片路径，送入 UNet 发光并画图"""

    # Cityscapes 的原生标注字典
    class_colors = {
        0: [128, 64, 128], 1: [244, 35, 232], 2: [70, 70, 70], 3: [102, 102, 156],
        4: [190, 153, 153], 5: [153, 153, 153], 6: [250, 170, 30], 7: [220, 220, 0],
        8: [107, 142, 35], 9: [152, 251, 152], 10: [70, 130, 180], 11: [220, 20, 60],
        12: [255, 0, 0], 13: [0, 0, 142], 14: [0, 0, 70], 15: [0, 60, 100],
        16: [0, 80, 100], 17: [0, 0, 230], 18: [119, 11, 32], 255: [0, 0, 0]
    }

    os.makedirs(save_dir, exist_ok=True)

    # 1. 完全复刻原装 Loader 的数据图片处理
    start_time = time.time()
    original_img = Image.open(image_path).convert('RGB')
    orig_w, orig_h = original_img.size # 记录原图长宽，推断完了要等比例放大回去

    # Resize -> 维度置换 -> 转 Float 再化至 [0, 1] 空间 (PIL 宽度在左，高度在右)
    resized_img = original_img.resize(target_size, Image.BILINEAR)
    img_tensor = torch.from_numpy(np.array(resized_img)).permute(2, 0, 1).float() / 255.0
    input_tensor = img_tensor.unsqueeze(0).to(device)

    # 2. 无梯度通过网络，推理出特征
    with torch.no_grad():
        output = model(input_tensor)
        # 把预测特征图暴降拉回真实城市街道拍摄照片的高宽（通常是 2048 x 1024）
        output = F.interpolate(output, size=(orig_h, orig_w), mode='bilinear', align_corners=False)
        end_time = time.time()
        print("纯输出时间",end_time - start_time)

    # 张量取所有类别的 Argmax 作为最大确信的标签索引 (H, W)
    pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

    # 3. 张量画布上色
    color_mask = np.zeros((orig_h, orig_w, 3), dtype=np.uint8)
    for label, color in class_colors.items():
        color_mask[pred_mask == label] = color

    color_mask_img = Image.fromarray(color_mask)

    # 4. 把原图和色彩掩盖相互叠加 (Alpha 指定半透率)
    blended = Image.blend(original_img, color_mask_img, alpha=0.55)

    # 5. 画图排版
    plt.figure(figsize=(18, 5))

    plt.subplot(1, 3, 1)
    plt.title("原始图像")
    plt.imshow(original_img)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title("UNet掩码")
    plt.imshow(color_mask_img)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title("叠加结果")
    plt.imshow(blended)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    # 6. 保存用于论文档留痕的高清图
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    blended.save(os.path.join(save_dir, f"blended_{timestamp}.png"))
    color_mask_img.save(os.path.join(save_dir, f"pure_mask_{timestamp}.png"))
    print(f"高清可视化结果已被写入至相对目录: [{save_dir}] 文件夹内")

# ======================= �� 实际调用区域 �� =======================

# 推理使用一张显卡就可以了，不要包装套娃 DataParallel
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
unet_eval_model = DPAI_UNet_Segmentation(num_classes=19, pretrained=False)

# 告诉它当时你训练时，保存 U-Net 成果对应的确切位置
# （如果在训练里因为各种代码结构调整找不到了，请手动替换掉后面的引号内容~）
ckpt_path = 'checkpoints_dpai_unet/dpai_unet/dpai_unet.pth'

# 载入显存
unet_eval_model = load_unet_model_weight(unet_eval_model, ckpt_path, device)

# 指定本地测试集的一张靓照
test_img_path = r'/home/ps/gzx/D2D_2.0/leftImg8bit/val/frankfurt/frankfurt_000000_000294_leftImg8bit.png'

# 开始推演
import time  # 导入时间模块
start_time = time.time()
predict_and_visualize(unet_eval_model, test_img_path, device)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"推理耗时: {elapsed_time:.2f} 秒")

In [ ]:
import torch
import numpy as np
import os
from torch.utils.data import DataLoader

# ======= 如果在你的 DPAI_Unet.ipynb 内跑，确保前面已经跑了 Dataset 类的定义 =======
# 从你的模块库中把其余 FPN 和 DeepLab 导进来
from model import ResNet50_AdaptiveInteraction_Segmentation
from model import DPAI_DeepLabV3P_Segmentation

def load_weight_safe(model, weight_path, device):
    """安全读取你在 ipynb 里跑出来的权重，自动脱壳多卡 'module.'"""
    if not os.path.exists(weight_path):
        raise FileNotFoundError(f"找不到权重文件: {weight_path}")
        
    state_dict = torch.load(weight_path, map_location=device)
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict, strict=True)
    model.to(device)
    model.eval()  # 关闭 Dropout 和非前向流
    return model

@torch.no_grad()
def evaluate_fwiou(model, val_loader, device):
    """根据你原有的混淆矩阵评价方式，加入频权 FWIoU 运算"""
    print("🚀 开始在验证集上推演混淆矩阵，请稍候...")
    confusion_matrix = np.zeros((19, 19))
    
    for batch_idx, (images, targets, _, _) in enumerate(val_loader):
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        
        preds = outputs.argmax(dim=1).cpu().numpy()
        targets = targets.cpu().numpy()
        
        # 逐像素装填矩阵，剔除 ignore_index (255)
        for t, p in zip(targets, preds):
            mask = (t != 255)
            label = 19 * t[mask].astype('int') + p[mask]
            confusion_matrix += np.bincount(label, minlength=19**2).reshape(19, 19)

        if (batch_idx + 1) % 20 == 0:
            print(f"  扫描进度: {batch_idx + 1} / {len(val_loader)} 批次")

    # -------------- 核心计算区域 --------------
    diag = np.diag(confusion_matrix)                     # 预测对的像素点
    gt_sum = confusion_matrix.sum(axis=1)                # 每一类的真实像素总和
    pred_sum = confusion_matrix.sum(axis=0)              # 每一类的预测像素总和
    
    # 1. 常规 mIoU 
    union = gt_sum + pred_sum - diag + 1e-10
    iou_per_class = diag / union
    miou = np.nanmean(iou_per_class)
    
    # 2. 计算频权 FWIoU: 该类所有像素量 / 数据集总合法像素量
    total_pixels = confusion_matrix.sum()
    freq = gt_sum / total_pixels                         # 算出频权 (频率权重)
    fwiou = (freq * iou_per_class).sum()                 # IoU 向频权加权后求和
    
    return miou, fwiou, confusion_matrix

# -------------- 初始化评估专用的 Dataset (共用) --------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
root_dir = r"E:\Laboratory files\code_project\city_data"  
# 把 batch 改成 8 或哪怕 16，评估不需要反向传播，显存占用远低于训练！
val_dataset = CityscapesDataset(root_dir, 'val')
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


In [ ]:
print("================================== 1. 评估 DPAI_Unet ==================================")
unet_model = DPAI_UNet_Segmentation(num_classes=19, pretrained=False)
# 对应 U-Net 的权重保存默认路径
unet_weight_path = 'checkpoints_dpai_unet/dpai_unet/dpai_unet.pth' 

unet_model = load_weight_safe(unet_model, unet_weight_path, device)
miou_u, fwiou_u, _ = evaluate_fwiou(unet_model, val_loader, device)

print(f"🏆 【DPAI_Unet 结论】")
print(f"     -> mIoU  平均交并比: {miou_u * 100:.2f} %")
print(f"     -> FWIoU 频权交并比: {fwiou_u * 100:.2f} %")


In [ ]:
print("\n================================== 2. 评估 DPAI_FPN ==================================")
fpn_model = ResNet50_AdaptiveInteraction_Segmentation(num_classes=19, pretrained=False)
# FPN 的默认历史权重路径
fpn_weight_path = 'checkpoints/DPAI_FPN.pth' 

fpn_model = load_weight_safe(fpn_model, fpn_weight_path, device)
miou_f, fwiou_f, _ = evaluate_fwiou(fpn_model, val_loader, device)

print(f"🏆 【DPAI_FPN 结论】")
print(f"     -> mIoU  平均交并比: {miou_f * 100:.2f} %")
print(f"     -> FWIoU 频权交并比: {fwiou_f * 100:.2f} %")


In [ ]:
print("\n================================== 3. 评估 DPAI_DeepLabV3+ ==================================")
deeplab_model = DPAI_DeepLabV3P_Segmentation(num_classes=19, pretrained=False)
# DeepLabV3+ 的保存路径
deeplab_weight_path = 'checkpoints_deeplab/DPAI_DeepLabV3P.pth' 

deeplab_model = load_weight_safe(deeplab_model, deeplab_weight_path, device)
miou_d, fwiou_d, _ = evaluate_fwiou(deeplab_model, val_loader, device)

print(f"🏆 【DPAI_DeepLabV3+ 结论】")
print(f"     -> mIoU  平均交并比: {miou_d * 100:.2f} %")
print(f"     -> FWIoU 频权交并比: {fwiou_d * 100:.2f} %")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_confusion_matrix(cm, classes, normalize=True, title='Confusion Matrix', save_name='confusion_matrix.png'):
    """
    专门为 19 类城市街景设计的顶级混淆矩阵绘制脚画
    """
    if normalize:
        # 对真值行进行归一化，使得矩阵呈现出每一个真实类别预测到了各列的百分比（行和为1）
        row_sums = cm.sum(axis=1)[:, np.newaxis]
        cm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums!=0)

    # 把图稍微拉长一点，防止 19 个类别标签文字挤在一起
    plt.figure(figsize=(16, 12))
    
    # 调起 Seaborn 画非常柔和的 'Blues' 蓝色渐变热图 
    # (如果 19x19 数据实在显得太挤妨碍美观可以设置 annot=False 关闭数值只看色块)
    sns.heatmap(cm, annot=True, cmap='Blues', fmt='.2f',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Accuracy Recall (Normalized)'})
                
    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.ylabel('True Class (Ground Truth Labels)', fontsize=15, fontweight='bold', labelpad=10)
    plt.xlabel('Predicted Class (Network Output Labels)', fontsize=15, fontweight='bold', labelpad=10)
    
    # 将 X 轴倾斜，完美适配比较长的 "traffic light" 英文分类
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)
    plt.tight_layout()
    
    # 保存高清大图供 Word 报告排版
    plt.savefig(save_name, dpi=300)
    print(f"🎉 你的高清混淆矩阵已经出炉保存为图像文件: [{save_name}] ！")
    plt.show()


# ===================== 👇 评估并立刻绘制 DPAI_Unet 的混淆图 👇 =====================

# Cityscapes 法定 19 类，千万不能错序
CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic light',
           'traffic sign', 'vegetation', 'terrain', 'sky', 'person', 'rider', 'car',
           'truck', 'bus', 'train', 'motorcycle', 'bicycle']

print("=========================== 【DPAI_Unet 混淆矩阵图生成】 ===========================")
unet_model = DPAI_UNet_Segmentation(num_classes=19, pretrained=False)
unet_model = load_weight_safe(unet_model, 'checkpoints_dpai_unet/dpai_unet/dpai_unet.pth', device)

# 因为有了我们刚才改的那句 return，现在它还能返回老三 cm_matrix 了！
miou_u, fwiou_u, unet_cm_matrix  = evaluate_fwiou(unet_model, val_loader, device)

print(f"🥇 【DPAI_Unet 结论】 mIoU: {miou_u * 100:.2f}% | FWIoU: {fwiou_u * 100:.2f}%")

# 直接把大矩阵扔给刚刚包装好的魔法函数
plot_confusion_matrix(unet_cm_matrix, CLASSES, 
                      title='DPAI U-Net Semantic Segmentation Confusion Matrix', 
                      save_name='DPAI_Unet_CM.png')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def merge_classes(cm, class_indices):
    """
    合并混淆矩阵中的多个类别为一个新类别

    参数:
        cm: 原始混淆矩阵
        class_indices: 要合并的类别索引列表

    返回:
        合并后的混淆矩阵
    """
    # 创建新的混淆矩阵
    n_classes = cm.shape[0] - len(class_indices) + 1
    new_cm = np.zeros((n_classes, n_classes))

    # 创建新索引映射
    new_index_map = {}
    new_idx = 0
    for i in range(cm.shape[0]):
        if i not in class_indices:
            new_index_map[i] = new_idx
            new_idx += 1
    # 添加合并类别的索引
    merged_idx = new_idx

    # 填充新的混淆矩阵
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i in class_indices and j in class_indices:
                # 两个都是要合并的类别
                new_cm[merged_idx, merged_idx] += cm[i, j]
            elif i in class_indices:
                # i是要合并的类别，j不是
                new_cm[merged_idx, new_index_map[j]] += cm[i, j]
            elif j in class_indices:
                # j是要合并的类别，i不是
                new_cm[new_index_map[i], merged_idx] += cm[i, j]
            else:
                # 两个都不是要合并的类别
                new_cm[new_index_map[i], new_index_map[j]] += cm[i, j]

    return new_cm

def plot_confusion_matrix(cm, classes, normalize=True, title='Confusion Matrix', save_name='confusion_matrix.png'):
    """
    专门为城市街景设计的顶级混淆矩阵绘制脚画
    """
    if normalize:
        # 对真值行进行归一化，使得矩阵呈现出每一个真实类别预测到了各列的百分比（行和为1）
        row_sums = cm.sum(axis=1)[:, np.newaxis]
        cm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums!=0)

    # 把图稍微拉长一点，防止类别标签文字挤在一起
    plt.figure(figsize=(16, 12))

    # 调起 Seaborn 画非常柔和的 'Blues' 蓝色渐变热图
    sns.heatmap(cm, annot=True, cmap='Blues', fmt='.2f',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': '准确率召回率'})

    plt.title(title, fontsize=20, fontweight='bold', pad=25)
    plt.ylabel('True Class (Ground Truth Labels)', fontsize=20, fontweight='bold', labelpad=20)
    plt.xlabel('Predicted Class (Model Output)', fontsize=20, fontweight='bold', labelpad=20)

    # 将 X 轴倾斜
    plt.xticks(rotation=45, ha='right', fontsize=15)
    plt.yticks(rotation=0, fontsize=15)
    plt.tight_layout()

    # 保存高清大图供 Word 报告排版
    plt.savefig(save_name, dpi=900)
    print(f"🎉 你的高清混淆矩阵已经出炉保存为图像文件: [{save_name}] ！")
    plt.show()

    # 计算并输出每个类别的评估指标
    print("\n" + "="*80)
    print(f"【{title}】各类别详细评估指标")
    print("="*80)

    # 计算每个类别的Precision, Recall和F1分数
    precision = np.diag(cm) / cm.sum(axis=0)
    recall = np.diag(cm) / cm.sum(axis=1)
    f1 = 2 * (precision * recall) / (precision + recall)

    # 处理除零情况
    precision = np.nan_to_num(precision)
    recall = np.nan_to_num(recall)
    f1 = np.nan_to_num(f1)

    # 创建DataFrame用于显示
    metrics_df = pd.DataFrame({
        '类别': classes,
        '准确率(Precision)': precision,
        '召回率(Recall)': recall,
        'F1分数': f1
    })

    # 格式化输出
    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)

    print(metrics_df)

    # 计算并输出整体指标
    print("\n" + "="*80)
    print(f"【{title}】整体评估指标")
    print("="*80)
    print(f"平均准确率 (Average Precision): {np.mean(precision):.4f}")
    print(f"平均召回率 (Average Recall): {np.mean(recall):.4f}")
    print(f"平均F1分数 (Average F1 Score): {np.mean(f1):.4f}")
    print("="*80 + "\n")


# ===================== 👇 评估并立刻绘制 DPAI_Unet 的混淆图 👇 =====================

# Cityscapes 法定 19 类，千万不能错序
CLASSES = ['road', 'sidewalk', 'building', 'wall', 'fence', 'pole', 'traffic light',
           'traffic sign', 'vegetation', 'terrain', 'sky', 'person', 'rider', 'car',
           'truck', 'bus', 'train', 'motorcycle', 'bicycle']

print("=========================== 【DPAI_Unet 混淆矩阵图生成】 ===========================")
unet_model = DPAI_UNet_Segmentation(num_classes=19, pretrained=False)
unet_model = load_weight_safe(unet_model, 'dpai_unet/dpai_unet.pth', device)

# 因为有了我们刚才改的那句 return，现在它还能返回老三 cm_matrix 了！
miou_u, fwiou_u, unet_cm_matrix  = evaluate_fwiou(unet_model, val_loader, device)

print(f"🥇 【DPAI_Unet 结论】 mIoU: {miou_u * 100:.2f}% | FWIoU: {fwiou_u * 100:.2f}%")

# 合并类别
# 第一次合并：5-7 (pole, traffic light, traffic sign) -> signs
merged_cm = merge_classes(unet_cm_matrix, [5, 6, 7])

# 第二次合并：13-16 (car, truck, bus, train) -> cars
# 注意：由于第一次合并后，原13-16的索引会变为12-15
merged_cm = merge_classes(merged_cm, [12, 13, 14, 15])

# 创建新的类别列表，确保与合并后的混淆矩阵对应
new_classes = []
for i in range(merged_cm.shape[0]):
    if i < 5:
        new_classes.append(CLASSES[i])
    elif i == 5:
        new_classes.append('signs')
    elif i < 12:
        new_classes.append(CLASSES[i-1])  # 由于合并了5-7，所以索引需要减1
    elif i == 12:
        new_classes.append('cars')
    else:
        new_classes.append(CLASSES[i-2])  # 由于合并了两组类别，所以索引需要减2

# 打印调试信息
print(f"混淆矩阵维度: {merged_cm.shape}")
print(f"类别列表长度: {len(new_classes)}")
print(f"类别列表: {new_classes}")

# 直接把大矩阵扔给刚刚包装好的魔法函数
plot_confusion_matrix(merged_cm, new_classes,
                      title='DPAI U-Net (Merged Classes)',
                      save_name='DPAI_Unet_CM_Merged.png')
